# CP3 · Notebook 04 — Tareas estructuradas

Pedirle al VLM 3 tareas concretas sobre cada imagen:
1. **Listar objetos críticos** (peatones, ciclistas, vehículos, semáforos, señales).
2. **Risk score 0-10**.
3. **Decisión recomendada** (continuar, frenar, ceder, ...).

~6 min.

In [ ]:
import time, json, re
from pathlib import Path
import torch
from PIL import Image
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm
import pandas as pd

DATA = Path('../datasets/dashcam-curated')
OUT = Path('../outputs')

MODEL_ID = 'vikhyatk/moondream2'
REVISION = '2024-08-26'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=REVISION, trust_remote_code=True,
                                               torch_dtype=torch.float32).eval()
print('modelo cargado')

## 1. Definir las 3 preguntas

In [ ]:
PROMPTS = {
    'objects':  'List all safety-critical objects visible in this driving scene (pedestrians, cyclists, vehicles, traffic signs, traffic lights). Use a numbered list, one per line.',
    'risk':     'On a scale from 0 (no risk) to 10 (immediate danger), what is the risk level of this driving scene? Respond with only the number first, then explain in one sentence.',
    'decision': 'What should the driver do right now? Choose one: CONTINUE, SLOW DOWN, YIELD, STOP. Then briefly justify.',
}

categories = ['trivial', 'urbano_standard', 'edge_visual', 'edge_semantic', 'trampa', 'ambigua']
all_imgs = [(cat, p) for cat in categories
             for p in sorted((DATA / cat).glob('*.jpg')) + sorted((DATA / cat).glob('*.png'))]
print(f'{len(all_imgs)} imágenes a procesar × 3 prompts = {3*len(all_imgs)} inferencias')

## 2. Inferir 3 preguntas por imagen

In [ ]:
results = []
for cat, img_path in tqdm(all_imgs, desc='structured'):
    img = Image.open(img_path).convert('RGB')
    enc = model.encode_image(img)
    row = {'category': cat, 'name': img_path.name}
    for tag, prompt in PROMPTS.items():
        t0 = time.perf_counter()
        ans = model.answer_question(enc, prompt, tokenizer)
        row[tag] = ans.strip()
        row[f'{tag}_time_s'] = time.perf_counter() - t0
    results.append(row)

print(f'\nLatencia total: {sum(r["objects_time_s"]+r["risk_time_s"]+r["decision_time_s"] for r in results):.0f}s')

## 3. Parsear risk score y action

In [ ]:
def parse_risk(text):
    m = re.search(r'\b(\d+(?:\.\d+)?)\b', text)
    if m: return float(m.group(1))
    return None

def parse_decision(text):
    text_up = text.upper()
    for choice in ['STOP', 'YIELD', 'SLOW DOWN', 'CONTINUE']:
        if choice in text_up: return choice
    return 'UNKNOWN'

for r in results:
    r['risk_score'] = parse_risk(r['risk'])
    r['action'] = parse_decision(r['decision'])
    r['n_objects_listed'] = len([l for l in r['objects'].splitlines() if l.strip() and any(c.isdigit() for c in l[:3])])

df = pd.DataFrame(results)
print(df[['category','name','risk_score','action','n_objects_listed']].to_string(index=False))

## 4. Agregados por categoría

In [ ]:
import matplotlib.pyplot as plt

by_cat = df.groupby('category').agg({
    'risk_score': ['mean', 'std'],
    'n_objects_listed': 'mean',
}).round(2)
print(by_cat)

# Plot risk by category
fig, ax = plt.subplots(figsize=(11, 4))
for cat, color in zip(categories, ['#4a6fa5','#f4a261','#DA4544','#e76f51','#9b5de5','#118ab2']):
    cat_data = df[df['category']==cat]
    if len(cat_data) and cat_data['risk_score'].notna().any():
        ax.scatter([cat]*len(cat_data), cat_data['risk_score'].dropna(), s=120, alpha=0.7, color=color, edgecolor='black')
ax.set_ylabel('Risk score (0-10) según VLM')
ax.set_xlabel('categoría')
ax.set_title('Risk score por categoría — Moondream2')
ax.set_ylim(-0.5, 10.5)
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=15)
plt.tight_layout(); plt.savefig(OUT / '04_risk_by_category.png', dpi=100, bbox_inches='tight'); plt.show()

**Lo que esperamos ver**:
- `trivial` con risk score bajo (0-3).
- `urbano_standard` medio (3-6).
- `edge_visual / edge_semantic / trampa` con risk altos (6-9).
- `ambigua` variable — depende de cómo el modelo interpreta.

**Si NO ves esa gradación**, el modelo no está distinguiendo categorías → relevante para tu análisis.

Guardar y pasar.

In [ ]:
df.to_csv(OUT / '04_structured_table.csv', index=False)
with open(OUT / '04_structured.json', 'w') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print(f'✅ Guardado en {OUT / "04_structured_table.csv"} y .json')
print('Ve a 05_consistencia.ipynb')